<a href="https://colab.research.google.com/github/Kownlage/ML/blob/main/maze_robot_lab_completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Graph Traversals — The Maze Robot

**Course:** CSCI 232 — Data Structures\
**Duration:** 1 Week

A robot must find its way from **S** to **G** on two maps: a warehouse floor with many
alternative routes, and a maze with exactly one route between any two cells. Reuse your
adjacency map, BFS and DFS to send one robot of each kind across both maps, watch them
search, and explain why they behave so differently.

After you paste your code, this notebook runs independently in Colab or locally.
Unfinished functions report PENDING rather than stopping Run All.

## Workflow

| Part | What you do |
|---|---|
| Part 1 | Paste your `adjacency_map.py` and `traversals.py` |
| Part 2 | Write `build_maze_graph`, `explore_until_goal` and `walking_distance`; watch your robots in the replay |
| Part 3 | Run the provided comparisons on both maps and explain the results |

Run the check cell after editing any function. Every group must pass:
**ALL CHECKS PASSED**.

### Rules

- Keep every given name, signature and docstring. A docstring states the cost and
  whether the input changes.
- Reuse your Part 1 code. Do not write your own BFS or DFS, and do not use
  `collections.deque` in Part 2.
- Use only the public methods of the graph, such as `insert_vertex`, `insert_edge` and
  `incident_edges`. Do not read attributes that start with `_`, such as `_outgoing`.
- A function must not change its input.

In [23]:
import inspect
import sys
import time
import types
from abc import ABC, abstractmethod
from html import escape
from typing import Dict, Generic, List, Sequence, Tuple, TypeVar

try:
    from dsa.graphs.base import Graph          # inside the course repository
except ImportError:
    # Colab, or a folder outside the repository: provide dsa/graphs/base.py here, so
    # that the line `from dsa.graphs.base import Graph` in your pasted code still works.
    V, E = TypeVar('V'), TypeVar('E')

    class Graph(ABC, Generic[V, E]):
        """The graph interface of dsa/graphs/base.py (Goodrich et al., Section 14.2)."""

        class Vertex(ABC):
            @abstractmethod
            def element(self): ...

            @abstractmethod
            def __eq__(self, other): ...

            @abstractmethod
            def __hash__(self): ...

        class Edge(ABC):
            @abstractmethod
            def element(self): ...

            @abstractmethod
            def endpoints(self): ...

            @abstractmethod
            def opposite(self, v): ...

            @abstractmethod
            def __eq__(self, other): ...

            @abstractmethod
            def __hash__(self): ...

        @abstractmethod
        def is_directed(self): ...

        @abstractmethod
        def vertex_count(self): ...

        @abstractmethod
        def edge_count(self): ...

        @abstractmethod
        def vertices(self): ...

        @abstractmethod
        def edges(self): ...

        @abstractmethod
        def get_edge(self, u, v): ...

        @abstractmethod
        def degree(self, v, outgoing=True): ...

        @abstractmethod
        def incident_edges(self, v, outgoing=True): ...

        @abstractmethod
        def insert_vertex(self, x=None): ...

        @abstractmethod
        def insert_edge(self, u, v, x=None): ...

        @abstractmethod
        def remove_vertex(self, v): ...

        @abstractmethod
        def remove_edge(self, e): ...

    for _module_name in ('dsa', 'dsa.graphs', 'dsa.graphs.base'):
        sys.modules.setdefault(_module_name, types.ModuleType(_module_name))
    sys.modules['dsa.graphs.base'].Graph = Graph


def _placeholder(name):
    """Stand in for code not pasted yet, so that Run All reports PENDING."""
    def unfinished(*args, **kwargs):
        raise NotImplementedError(name)
    unfinished.__name__ = name
    return unfinished


for _name in ('AdjacencyMapGraph', 'dfs', 'dfs_paths', 'bfs', 'bfs_paths', 'construct_path'):
    globals().setdefault(_name, _placeholder(_name))  # your pasted code replaces these
print('Setup complete. Continue from top to bottom.')

Setup complete. Continue from top to bottom.


## Part 1 — Paste your graph code

Paste your own completed code into the two empty cells below. Do not rewrite it or change
it: Parts 2 and 3 call these functions exactly as you pasted them.

| Cell | Paste the full contents of | What it defines |
|---|---|---|
| First | `dsa/graphs/adjacency_map.py` | `AdjacencyMapGraph` |
| Second | `dsa/graphs/traversals.py` | `dfs`, `dfs_paths`, `bfs`, `bfs_paths`, `construct_path` |

Keep the line `from dsa.graphs.base import Graph` at the top of each file. The setup cell
makes it work in Colab too.

In [24]:
"""Adjacency map implementation of a graph."""
from typing import TypeVar, Iterator, Optional, Tuple

V = TypeVar('V')
E = TypeVar('E')


class AdjacencyMapGraph(Graph[V, E]):
    """Graph implementation using an adjacency map.

    Each vertex maintains a dictionary mapping adjacent vertices to
    the edges connecting them. This provides O(1) edge lookup between
    any two vertices.

    Supports both directed and undirected graphs.
    """

    class _Vertex(Graph.Vertex):
        """Vertex implementation for AdjacencyMapGraph."""
        __slots__ = '_element'

        def __init__(self, element: V):
            self._element = element

        def element(self) -> V:
            return self._element

        def __eq__(self, other: object) -> bool:
            return self is other

        def __hash__(self) -> int:
            return hash(id(self))

    class _Edge(Graph.Edge):
        """Edge implementation for AdjacencyMapGraph."""
        __slots__ = '_element', '_origin', '_destination'

        def __init__(self, origin: 'AdjacencyMapGraph._Vertex',
                     destination: 'AdjacencyMapGraph._Vertex',
                     element: E):
            self._element = element
            self._origin = origin
            self._destination = destination

        def element(self) -> E:
            return self._element

        def endpoints(self) -> Tuple[Graph.Vertex, Graph.Vertex]:
            return (self._origin, self._destination)

        def opposite(self, v: Graph.Vertex) -> Graph.Vertex:
            if v is self._origin:
                return self._destination
            elif v is self._destination:
                return self._origin
            else:
                raise ValueError('v is not incident to this edge')

        def __eq__(self, other: object) -> bool:
            return self is other

        def __hash__(self) -> int:
            return hash(id(self))

    def __init__(self, directed: bool = False):
        """Create an empty graph.

        Args:
            directed: True for a directed graph, False for undirected.
        """
        self._outgoing = {}
        self._incoming = {} if directed else self._outgoing
        self._edge_count = 0

    def _validate_vertex(self, v: Graph.Vertex) -> '_Vertex':
        """Verify that v is a Vertex of this graph and return it."""
        if not isinstance(v, self._Vertex):
            raise TypeError('Vertex must be proper Vertex type')
        if v not in self._outgoing:
            raise ValueError('Vertex does not belong to this graph')
        return v

    def is_directed(self) -> bool:
        return self._incoming is not self._outgoing

    def vertex_count(self) -> int:
        return len(self._outgoing)

    def edge_count(self) -> int:
        return self._edge_count

    def vertices(self) -> Iterator[Graph.Vertex]:
        return iter(self._outgoing.keys())

    def edges(self) -> Iterator[Graph.Edge]:
        seen = set()
        for secondary_map in self._outgoing.values():
            seen.update(secondary_map.values())
        return iter(seen)

    def get_edge(self, u: Graph.Vertex, v: Graph.Vertex) -> Optional[Graph.Edge]:
        self._validate_vertex(u)
        self._validate_vertex(v)
        return self._outgoing[u].get(v)

    def degree(self, v: Graph.Vertex, outgoing: bool = True) -> int:
        self._validate_vertex(v)
        adj = self._outgoing if outgoing else self._incoming
        return len(adj[v])

    def incident_edges(self, v: Graph.Vertex, outgoing: bool = True) -> Iterator[Graph.Edge]:
        self._validate_vertex(v)
        adj = self._outgoing if outgoing else self._incoming
        return iter(adj[v].values())

    def insert_vertex(self, x: V = None) -> Graph.Vertex:
        v = self._Vertex(x)
        self._outgoing[v] = {}
        if self.is_directed():
            self._incoming[v] = {}
        return v

    def insert_edge(self, u: Graph.Vertex, v: Graph.Vertex, x: E = None) -> Graph.Edge:
        self._validate_vertex(u)
        self._validate_vertex(v)
        if self.get_edge(u, v) is not None:
            raise ValueError('u and v are already adjacent')
        e = self._Edge(u, v, x)
        self._outgoing[u][v] = e
        self._incoming[v][u] = e
        self._edge_count += 1
        return e

    def remove_vertex(self, v: Graph.Vertex) -> V:
        self._validate_vertex(v)
        for e in list(self._outgoing[v].values()):
            self.remove_edge(e)
        if self.is_directed():
            for e in list(self._incoming[v].values()):
                self.remove_edge(e)
            del self._incoming[v]
        del self._outgoing[v]
        return v.element()

    def remove_edge(self, e: Graph.Edge) -> E:
        if not isinstance(e, self._Edge):
            raise TypeError('Edge must be proper Edge type')
        u, v = e.endpoints()
        self._validate_vertex(u)
        self._validate_vertex(v)
        if self._outgoing[u].get(v) is not e:
            raise ValueError('Edge does not belong to this graph')
        del self._outgoing[u][v]
        del self._incoming[v][u]
        self._edge_count -= 1
        return e.element()

In [25]:
"""Graph traversal algorithms."""

from collections import deque
from typing import TypeVar, Dict, Set, Iterator, Optional

V = TypeVar('V')
E = TypeVar('E')


def dfs(graph: Graph[V, E], start: Graph.Vertex) -> Iterator[Graph.Vertex]:
    """Generate vertices reachable from start in depth-first order.

    Depth-first search explores as far as possible along each branch
    before backtracking.

    Args:
        graph: The graph to traverse.
        start: The starting vertex.

    Yields:
        Vertices in depth-first order.
    """
    visited: Set[Graph.Vertex] = set()

    def _dfs(u: Graph.Vertex) -> Iterator[Graph.Vertex]:
        visited.add(u)
        yield u
        for edge in graph.incident_edges(u):
            v = edge.opposite(u)
            if v not in visited:
                yield from _dfs(v)

    return _dfs(start)


def dfs_paths(graph: Graph[V, E], start: Graph.Vertex) -> Dict[Graph.Vertex, Graph.Edge]:
    """Return a dictionary mapping vertices to discovery edges.

    The discovery edges form a DFS tree rooted at start. For each
    reachable vertex v (except start), the dictionary maps v to the
    edge used to discover v.

    Args:
        graph: The graph to traverse.
        start: The starting vertex.

    Returns:
        Dictionary mapping discovered vertices to their discovery edges.
    """
    discovered: Dict[Graph.Vertex, Optional[Graph.Edge]] = {start: None}
    stack = [start]

    while stack:
        u = stack.pop()
        for edge in reversed(list(graph.incident_edges(u))):
            v = edge.opposite(u)
            if v not in discovered:
                discovered[v] = edge
                stack.append(v)

    del discovered[start]
    return discovered  # type: ignore


def bfs(graph: Graph[V, E], start: Graph.Vertex) -> Iterator[Graph.Vertex]:
    """Generate vertices reachable from start in breadth-first order.

    Breadth-first search explores all neighbors at the present depth
    before moving to vertices at the next depth level.

    Args:
        graph: The graph to traverse.
        start: The starting vertex.

    Yields:
        Vertices in breadth-first order.
    """
    visited: Set[Graph.Vertex] = {start}
    queue = deque([start])

    while queue:
        u = queue.popleft()
        yield u
        for edge in graph.incident_edges(u):
            v = edge.opposite(u)
            if v not in visited:
                visited.add(v)
                queue.append(v)


def bfs_paths(graph: Graph[V, E], start: Graph.Vertex) -> Dict[Graph.Vertex, Graph.Edge]:
    """Return a dictionary mapping vertices to discovery edges.

    The discovery edges form a BFS tree rooted at start, which contains
    shortest paths (by number of edges) from start to each reachable vertex.

    Args:
        graph: The graph to traverse.
        start: The starting vertex.

    Returns:
        Dictionary mapping discovered vertices to their discovery edges.
    """
    discovered: Dict[Graph.Vertex, Optional[Graph.Edge]] = {start: None}
    queue = deque([start])

    while queue:
        u = queue.popleft()
        for edge in graph.incident_edges(u):
            v = edge.opposite(u)
            if v not in discovered:
                discovered[v] = edge
                queue.append(v)

    del discovered[start]
    return discovered  # type: ignore


def construct_path(start: Graph.Vertex, end: Graph.Vertex,
                   discovered: Dict[Graph.Vertex, Graph.Edge]) -> list:
    """Reconstruct path from start to end using discovered edges.

    Args:
        start: The starting vertex.
        end: The ending vertex.
        discovered: Dictionary from dfs_paths or bfs_paths.

    Returns:
        List of vertices forming the path from start to end,
        or an empty list if no path exists.
    """
    if start == end:
        return [start]
    if end not in discovered:
        return []

    path = []
    curr = end
    while curr != start:
        path.append(curr)
        edge = discovered[curr]
        curr = edge.opposite(curr)
    path.append(start)
    path.reverse()
    return path

## Part 2 — Build the Maze Graph and Explore

### Scenario: Two Robots, Two Maps

A delivery robot must travel from **S** to **G**. Two robots run different searches: the
**BFS robot** explores cells in order of their distance from S, and the **DFS robot** goes as
deep as it can along one corridor before it backs up. Both work on the two maps printed by
the next cell, each 15 × 31 cells:

- **Map A, the warehouse:** open aisles between shelves, so there are many alternative
  routes between two cells.
- **Map B, the maze:** narrow corridors, with exactly one route between any two cells.

`#` is a wall. Every other character is an open cell the robot can stand on. The robot
moves one step at a time, north, south, east or west, never diagonally.

### Tasks

| Function | What it returns | Changes its input? |
|---|---|---|
| `build_maze_graph(grid)` | `(graph, cells)`: an undirected graph of the open cells, and a dict from `(row, col)` to each vertex | No |
| `explore_until_goal(graph, start, goal, method)` | The cells visited in order, from `start` up to and including `goal` | No |
| `walking_distance(graph, order)` | How many steps a robot walks to visit the cells of `order`, one after another | No |

The next cell provides the two maps and `find_cell`. Do not edit it.

In [26]:
MAPS = {
    'warehouse': [
        '###############################',
        '#S............................#',
        '#.####..####..####..####..###.#',
        '#.............................#',
        '#.####..####..####..####..###.#',
        '#G............................#',
        '#.####..####..####..####..###.#',
        '#.............................#',
        '#.####..####..####..####..###.#',
        '#.............................#',
        '#.####..####..####..####..###.#',
        '#.............................#',
        '#.####..####..####..####..###.#',
        '#.............................#',
        '###############################',
    ],
    'maze': [
        '###############################',
        '#S#.....#.........#.#.....#...#',
        '#.#.###.#.#######.#.#.#.#.###.#',
        '#...#.#.#...#.....#...#.#.....#',
        '#####.#.#.#.#####.#.###.#######',
        '#...#.#.#.#.....#.#...#.......#',
        '#.#.#.#.#.#####.#.#####.#####.#',
        '#.#...#.#.#.....#.....#.....#.#',
        '#.#####.#.#.#########.#######.#',
        '#.#.....#.#.....#...#.......#.#',
        '#.#.###########.#.#########.#.#',
        '#.#.#.........#.#.....#...#...#',
        '#.#.#.#######.#.#.#.###.#.###.#',
        '#.....#.........#.#.....#....G#',
        '###############################',
    ],
}
MAP_TITLES = {'warehouse': 'Map A: warehouse', 'maze': 'Map B: maze'}


def find_cell(grid: Sequence[str], char: str) -> Tuple[int, int]:
    """Provided. Return the (row, col) of the first char in grid, such as 'S' or 'G'. O(rows * cols)."""
    for row, line in enumerate(grid):
        col = line.find(char)
        if col != -1:
            return row, col
    raise ValueError(f'{char!r} is not in the grid')


for _name, _grid in MAPS.items():
    print(f"{MAP_TITLES[_name]}: S at {find_cell(_grid, 'S')}, G at {find_cell(_grid, 'G')}")
    print('\n'.join(_grid), end='\n\n')

Map A: warehouse: S at (1, 1), G at (5, 1)
###############################
#S............................#
#.####..####..####..####..###.#
#.............................#
#.####..####..####..####..###.#
#G............................#
#.####..####..####..####..###.#
#.............................#
#.####..####..####..####..###.#
#.............................#
#.####..####..####..####..###.#
#.............................#
#.####..####..####..####..###.#
#.............................#
###############################

Map B: maze: S at (1, 1), G at (13, 29)
###############################
#S#.....#.........#.#.....#...#
#.#.###.#.#######.#.#.#.#.###.#
#...#.#.#...#.....#...#.#.....#
#####.#.#.#.#####.#.###.#######
#...#.#.#.#.....#.#...#.......#
#.#.#.#.#.#####.#.#####.#####.#
#.#...#.#.#.....#.....#.....#.#
#.#####.#.#.#########.#######.#
#.#.....#.#.....#...#.......#.#
#.#.###########.#.#########.#.#
#.#.#.........#.#.....#...#...#
#.#.#.#######.#.#.#.###.#.###.#
#.....#.........#.#.

**2.1 Build the Maze Graph**

Turn a map into a graph that your `bfs` and `dfs` can search:

- Each open cell `(row, col)` becomes **one vertex** whose element is `(row, col)`.
- Two open cells that share a side are joined by **exactly one edge**. Your `insert_edge`
  raises `ValueError` if the same two cells are joined twice.
- `cells[(row, col)]` is the vertex of that cell. Your searches take vertices, not
  coordinates, so this dict is how you find the vertices of S and G.

Visit the cells row by row, left to right. When you reach a cell, the open cell above it and
the open cell on its left already have vertices, so join the new cell to the cell above
first, then to the cell on its left. Follow this order exactly: the check cell tests it, and
Part 3 asks what it changes.

Example: `["S.", "#G"]` has 3 open cells and 2 edges.

In [27]:
def build_maze_graph(grid: Sequence[str]) -> Tuple[Graph, Dict[Tuple[int, int], Graph.Vertex]]:
    """Return (graph, cells) for the maze in grid.

    '#' is a wall; every other character ('.', 'S', 'G') is an open cell.
    Each open cell (r, c) becomes one vertex whose element is (r, c), and
    cells[(r, c)] is that vertex. Visit the cells row by row, left to right,
    and join each new cell to the open cell above it, then to the open cell
    on its left.

    O(rows * cols). Does NOT modify grid.
    """
    graph = AdjacencyMapGraph()
    cells = {}
    for r, line in enumerate(grid):
        for c, char in enumerate(line):
            if char == "#":
                continue
            vertex = graph.insert_vertex((r, c))
            cells[(r, c)] = vertex
            if (r - 1, c) in cells:
                graph.insert_edge(cells[(r - 1, c)], vertex)
            if (r, c - 1) in cells:
                graph.insert_edge(cells[(r, c - 1)], vertex)
    return graph, cells


**2.2 Explore Until the Goal**

`method` is `'bfs'` or `'dfs'`. Call your `bfs` or `dfs` from Part 1, collect the cells it
visits **one at a time**, and stop as soon as you reach `goal`:

```python
order = explore_until_goal(graph, start, goal, 'dfs')
```

Requirements:

- Raise `ValueError` for any other `method`.
- Stop as soon as `goal` appears, so that no cell after it is explored. Your `bfs` and `dfs`
  are generators: a `for` loop receives one cell at a time, and `break` stops the search.
- If `goal` cannot be reached, return every cell the search visits.
- If `start` is `goal`, return `[start]`.

*Limit:* a recursive `dfs` calls itself once for each step deeper. On an open floor of
60 × 60 cells it goes 3,599 calls deep, past Python's recursion limit, and raises
`RecursionError`. The two maps in this lab have at most 263 cells, so they are safe.

In [28]:
def explore_until_goal(graph: Graph, start: Graph.Vertex, goal: Graph.Vertex,
                       method: str) -> List[Graph.Vertex]:
    """Return the cells visited from start, in order, up to and including goal.

    method is 'bfs' or 'dfs': use bfs() or dfs() from Part 1. Stop as soon as
    you reach goal. If goal cannot be reached, return every visited cell.

    O(n + m). Does NOT modify graph.

    Raises:
        ValueError: if method is not 'bfs' or 'dfs'.
    """
    if method not in ("bfs", "dfs"):
        raise ValueError("method must be 'bfs' or 'dfs'")
    visits = bfs(graph, start) if method == "bfs" else dfs(graph, start)
    order = []
    for vertex in visits:
        order.append(vertex)
        if vertex is goal:
            break
    return order


**2.3 Walking Distance**

A real robot cannot jump. To explore the cells of `order` one after another, it must walk
from each cell to the next. `walking_distance` adds up those walks:

- From each cell to the next one in `order`, the robot takes a shortest route. Find it with
  `bfs_paths` and `construct_path` from Part 1.
- A route of k cells is k − 1 steps. An `order` with fewer than two cells needs 0 steps.

Example on `["S.", "#G"]`: walking the order S, (0, 1), G takes 1 + 1 = 2 steps, but walking
S, G, (0, 1) takes 2 + 1 = 3 steps, because the robot must pass (0, 1) to reach G.

*Simplification:* the robot takes the shortest route on the whole map, as if it already knew
the cells it has not explored yet.

In [29]:
def walking_distance(graph: Graph, order: Sequence[Graph.Vertex]) -> int:
    """Return how many steps a robot walks to visit the cells of order, in order.

    From each cell to the next one in order, the robot takes a shortest route:
    find it with bfs_paths and construct_path from Part 1. A route of k cells
    takes k - 1 steps. order comes from explore_until_goal, so every cell can
    be reached.

    O(k (n + m)) for k cells. Does NOT modify graph or order.
    """
    total = 0
    for i in range(len(order) - 1):
        here = order[i]
        there = order[i + 1]
        route = construct_path(here, there, bfs_paths(graph, here))
        total += len(route) - 1
    return total


**Replay: watch your robots**

Run the next two cells after you finish Part 2. The replay calls **your own** code: it builds
the map with `build_maze_graph`, runs `explore_until_goal` with `'bfs'` and with `'dfs'`, and
measures each walk with `walking_distance`. Both robots start together and explore one new
cell per step. A robot that reaches G stops and shows its route, while the other keeps
searching. Optional `ipywidgets` gives the controls.

1. Choose a map.
2. Choose a speed: **0.5×**, **1×** or **2×**. At 1× the robots explore 8 cells a second.
3. Click the play button (▶) to start and pause (❚❚) to stop, or drag the slider to any step.
4. After changing your code, rerun its cell, then rerun the dashboard cell.

Things to look for (no written answer is required):

- On **Map A**, which robot reaches G first? Which way does the DFS robot turn at S?
- On **Map B**, compare the two green routes once both robots have finished.
- Watch **walked so far**. Why does it often jump by many steps for the BFS robot, but
  rarely for the DFS robot?

Without widgets, run `show_robot_replay('maze', step=40)` in a new cell and change `step`.
The second cell shows the last step on both maps without controls.

In [30]:
ROBOT_SPEEDS = {'0.5×': 250, '1×': 125, '2×': 62}   # milliseconds per step; 1× is 8 cells a second
CELL_PX = 16
MAZE_COLORS = {'wall': '#31475d', 'open': '#f7f8fa', 'route': '#1baf7a', 'robot': '#eb6834',
               'walk': '#d99a1e', 'early': (214, 230, 245), 'late': (74, 128, 186)}
MAZE_CSS = """
<style>
.maze-glass{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Arial,sans-serif;color:#17202a;
 background:radial-gradient(ellipse at top left,#fff,transparent 65%),linear-gradient(135deg,#edf0f3,#dce2e8);
 border:1px solid #fff;border-radius:22px;padding:18px 20px;box-shadow:0 12px 32px #16223320;
 line-height:1.45;max-width:1080px;margin-bottom:12px}
.maze-glass *{box-sizing:border-box}.maze-glass h2{font-size:21px;margin:0 0 6px;color:#17202a}
.maze-glass h3{font-size:15px;margin:0 0 6px;color:#17202a}
.maze-glass .chip{display:inline-block;background:#f7f8fa;border:1px solid #c4ccd4;border-radius:30px;
 padding:3px 10px;margin:2px 4px 4px 0;font-size:12px;font-weight:650}
.maze-glass .card{background:#ffffffd9;border:1px solid #fff;border-radius:14px;padding:12px 14px;
 margin:10px 0;box-shadow:0 6px 20px #17202a0a}
.maze-glass .pair{display:flex;flex-wrap:wrap;gap:12px}
.maze-glass .pair .card{flex:1 1 440px;min-width:0;margin:6px 0}
.maze-glass .muted{color:#5a6775;font-size:12px}
.maze-glass .banner{border-left:4px solid #31475d;padding:8px 12px;background:#eef3f7;border-radius:8px;
 font-size:13px;margin:8px 0}.maze-glass .banner.bad{border-color:#991c3d;background:#fff0f2}
.maze-glass .banner.warn{border-color:#9b6a16;background:#fff7e5}
.maze-glass svg{display:block;width:100%;height:auto;margin-top:6px}
</style>
"""


def _trace_problem(grid, traces):
    """Describe the first impossible thing in what your robots did, or return None."""
    start, goal = find_cell(grid, 'S'), find_cell(grid, 'G')
    for method, trace in traces.items():
        order, label = trace['order'], f'{method.upper()} robot'
        if not order or order[0] != start:
            return f'{label}: explore_until_goal must start at S {start}'
        if len(set(order)) != len(order):
            return f'{label}: a cell appears twice in the order it explored'
        if goal in order and order[-1] != goal:
            return f'{label}: the search went on after reaching G; stop with break'
        for here, there, steps, walk in zip(order, order[1:], trace['steps'], trace['walks']):
            if steps != len(walk) - 1:
                return (f'{label}: walking_distance gives {steps} steps from {here} to {there}, '
                        f'but the shortest route has {len(walk) - 1}')
    return None


def record_robots(name):
    """Run your Part 1 and Part 2 code on MAPS[name] for both robots.

    Returns (status, traces). status is 'ok', 'PENDING: ...' or a problem found in your
    results; traces maps 'bfs' and 'dfs' to the cells that robot explored, the steps it
    walked between them, and its route from S to G.
    """
    grid = MAPS[name]
    try:
        graph, cells = build_maze_graph(grid)
        start, goal = cells[find_cell(grid, 'S')], cells[find_cell(grid, 'G')]
        traces = {}
        for method in ('bfs', 'dfs'):
            order = explore_until_goal(graph, start, goal, method)
            steps, walks = [], []
            for here, there in zip(order, order[1:]):
                steps.append(walking_distance(graph, [here, there]))
                walks.append([cell.element() for cell in construct_path(here, there, bfs_paths(graph, here))])
            paths = bfs_paths(graph, start) if method == 'bfs' else dfs_paths(graph, start)
            traces[method] = {'order': [cell.element() for cell in order], 'steps': steps, 'walks': walks,
                              'route': [cell.element() for cell in construct_path(start, goal, paths)]}
    except NotImplementedError as exc:
        return f'PENDING: finish {exc or "your pasted Part 1 code"}', {}
    except Exception as exc:
        return f'Your code raised {type(exc).__name__}: {exc}', {}
    return _trace_problem(grid, traces) or 'ok', traces


def robot_frame_count(traces):
    """Steps in a replay: until the robot that explores the most cells is done."""
    return max((len(trace['order']) for trace in traces.values()), default=1)


def _shade(fraction):
    early, late = MAZE_COLORS['early'], MAZE_COLORS['late']
    return '#' + ''.join(f'{round(a + (b - a) * fraction):02x}' for a, b in zip(early, late))


def _robot_svg(grid, trace, step):
    rows, cols, px = len(grid), max(len(line) for line in grid), CELL_PX
    order = trace['order'] if trace else []
    shown = order[:step + 1]
    seen = {cell: index for index, cell in enumerate(shown)}
    reached = bool(order) and step >= len(order) - 1 and order[-1] == find_cell(grid, 'G')
    route = set(trace['route']) if reached else set()
    last = max(len(order) - 1, 1)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {cols * px} {rows * px}" role="img" '
             'aria-label="The map at this replay step" font-family="Arial, sans-serif">',
             f'<rect width="{cols * px}" height="{rows * px}" fill="{MAZE_COLORS["wall"]}"/>']
    for r, line in enumerate(grid):
        for c, char in enumerate(line):
            if char == '#':
                continue
            if (r, c) in route:
                fill = MAZE_COLORS['route']
            elif (r, c) in seen:
                fill = _shade(seen[(r, c)] / last)
            else:
                fill = MAZE_COLORS['open']
            parts.append(f'<rect x="{c * px + 0.5}" y="{r * px + 0.5}" width="{px - 1}" '
                         f'height="{px - 1}" fill="{fill}"/>')
    if trace and 1 <= step < len(order):         # the walk that brought the robot to its newest cell
        points = ' '.join(f'{c * px + px / 2},{r * px + px / 2}' for r, c in trace['walks'][step - 1])
        parts.append(f'<polyline points="{points}" fill="none" stroke="{MAZE_COLORS["walk"]}" '
                     'stroke-width="3" stroke-linejoin="round" stroke-linecap="round" opacity="0.9"/>')
    if shown:
        r, c = shown[-1]
        parts.append(f'<circle cx="{c * px + px / 2}" cy="{r * px + px / 2}" r="{px * 0.38}" '
                     f'fill="{MAZE_COLORS["robot"]}" stroke="#fff" stroke-width="2"/>')
    for char in 'SG':
        r, c = find_cell(grid, char)
        parts.append(f'<text x="{c * px + px / 2}" y="{r * px + px * 0.75}" text-anchor="middle" '
                     f'font-size="12" font-weight="800" fill="#17202a">{char}</text>')
    parts.append('</svg>')
    return ''.join(parts)


def _robot_card(grid, method, trace, step):
    title = f'{method.upper()} robot'
    if not trace:
        return f'<div class="card"><h3>{title}</h3>{_robot_svg(grid, None, step)}</div>'
    order = trace['order']
    shown = min(step + 1, len(order))
    walked = sum(trace['steps'][:shown - 1])
    if step < len(order) - 1:
        status = 'searching…'
    elif order[-1] == find_cell(grid, 'G'):
        status = f'reached G after exploring {len(order)} cells · route: {len(trace["route"]) - 1} steps'
    else:
        status = 'explored every cell it can reach: G is unreachable'
    chips = ''.join(f'<span class="chip">{text}</span>'
                    for text in (f'explored: {shown}', f'walked so far: {walked}', status))
    return f'<div class="card"><h3>{title}</h3>{chips}{_robot_svg(grid, trace, step)}</div>'


def render_robots(name, status, traces, step):
    """HTML for one replay step of both robots on MAPS[name]."""
    grid = MAPS[name]
    frames = robot_frame_count(traces)
    step = max(0, min(step, frames - 1))
    banner = ''
    if status.startswith('PENDING'):
        banner = f'<div class="banner warn">{escape(status)}</div>'
    elif status != 'ok':
        banner = f'<div class="banner bad">{escape(status)}</div>'
    cards = ''.join(_robot_card(grid, method, traces.get(method), step) for method in ('bfs', 'dfs'))
    return (f'<div class="maze-glass"><h2>Replay · {MAP_TITLES[name]}: your BFS robot and your DFS robot</h2>'
            f'{banner}<span class="chip">step {step + 1} of {frames}</span><div class="pair">{cards}</div>'
            '<p class="muted">Dark: wall. Blue: explored; darker cells were explored later. Orange dot: '
            'the robot. Gold line: the walk that brought the robot to its newest cell. Green: the route '
            'from S to G, shown once the robot reaches G. At each step, every robot that is still '
            'searching explores one new cell.</p></div>')


def _display_html(html):
    try:
        from IPython.display import HTML, display
    except ImportError:
        return html
    else:
        display(HTML(html))


def show_robot_replay(name, step=None):
    """Without widgets: show one step of the replay on MAPS[name]; by default, its last step."""
    status, traces = record_robots(name)
    frames = robot_frame_count(traces)
    step = frames - 1 if step is None else step
    return _display_html(MAZE_CSS + render_robots(name, status, traces, step))


def make_robot_dashboard():
    """Choose a map and a speed, then play both robots step by step."""
    try:
        import ipywidgets as widgets
        from IPython.display import display
    except ImportError:
        print('Optional controls: install ipywidgets and rerun this cell. Without it, run '
              "show_robot_replay('maze', step=40) in a new cell.")
        return None
    chosen = widgets.ToggleButtons(options=[(title, name) for name, title in MAP_TITLES.items()],
                                   description='Map:')
    chosen.style.button_width = '150px'
    speed = widgets.ToggleButtons(options=list(ROBOT_SPEEDS), value='1×', description='Speed:')
    speed.style.button_width = '60px'
    play = widgets.Play(value=0, min=0, max=0, step=1, interval=ROBOT_SPEEDS['1×'], show_repeat=False)
    slider = widgets.IntSlider(value=0, min=0, max=0, description='Step:',
                               layout=widgets.Layout(width='620px'))
    widgets.jslink((play, 'value'), (slider, 'value'))
    panel = widgets.HTML()
    state = {}

    def draw(*_):
        panel.value = MAZE_CSS + render_robots(chosen.value, state['status'], state['traces'], slider.value)

    def load(*_):
        state['status'], state['traces'] = record_robots(chosen.value)
        play.playing = False
        play.value = slider.value = 0
        play.max = slider.max = robot_frame_count(state['traces']) - 1
        draw()

    def change_speed(change):
        play.interval = ROBOT_SPEEDS[change['new']]

    chosen.observe(load, names='value')
    speed.observe(change_speed, names='value')
    slider.observe(draw, names='value')
    app = widgets.VBox([chosen, widgets.HBox([play, speed]), slider, panel])
    load()
    display(app)
    return app

In [31]:
robot_app = make_robot_dashboard()

In [32]:
show_robot_replay('warehouse')
show_robot_replay('maze')

## Part 3 — BFS vs. DFS on the Two Maps

Part 3 runs your robots on both maps and asks you to explain what you see. The next cell
contains the provided helpers; do not edit it. You write no code in this part.

| Helper | What it does |
|---|---|
| `maze_profile(name)` | Counts the open cells n, the edges m, the loops m − n + 1, the dead ends and the junctions of a map |
| `print_profiles()` | Prints `maze_profile` for both maps |
| `robot_run(name, method, swap=False)` | Runs one robot with your functions and returns its route length, cells explored and walking distance |
| `compare_robots(name, swap=False)` | Prints `robot_run` for both robots; `swap=True` exchanges S and G |

In [33]:
def maze_profile(name):
    """Return n, m, loops, dead ends and junctions of MAPS[name], using your build_maze_graph."""
    graph, _ = build_maze_graph(MAPS[name])
    n, m = graph.vertex_count(), graph.edge_count()
    degrees = [graph.degree(cell) for cell in graph.vertices()]
    return {'n': n, 'm': m, 'loops': m - n + 1,
            'dead ends': degrees.count(1), 'junctions': sum(d >= 3 for d in degrees)}


def print_profiles():
    """Print maze_profile for both maps."""
    print(f"{'Map':<18}{'n':>6}{'m':>6}{'m - n + 1':>11}{'dead ends':>11}{'junctions':>11}")
    for name in MAPS:
        try:
            p = maze_profile(name)
        except NotImplementedError as exc:
            print(f'{MAP_TITLES[name]:<18}PENDING: finish {exc or "your pasted Part 1 code"}')
            continue
        print(f"{MAP_TITLES[name]:<18}{p['n']:>6}{p['m']:>6}{p['loops']:>11}"
              f"{p['dead ends']:>11}{p['junctions']:>11}")
    print()


def robot_run(name, method, swap=False):
    """Run one robot on MAPS[name] with your functions: route length, cells explored, steps walked."""
    grid = MAPS[name]
    graph, cells = build_maze_graph(grid)
    start, goal = cells[find_cell(grid, 'S')], cells[find_cell(grid, 'G')]
    if swap:
        start, goal = goal, start
    order = explore_until_goal(graph, start, goal, method)
    paths = bfs_paths(graph, start) if method == 'bfs' else dfs_paths(graph, start)
    route = construct_path(start, goal, paths)
    return {'route': len(route) - 1, 'explored': len(order), 'walked': walking_distance(graph, order)}


def compare_robots(name, swap=False):
    """Print the route length, cells explored and steps walked of both robots on MAPS[name]."""
    start, goal = find_cell(MAPS[name], 'S'), find_cell(MAPS[name], 'G')
    if swap:
        start, goal = goal, start
    print(f'{MAP_TITLES[name]}: from {start} to {goal}' + ('  (S and G swapped)' if swap else ''))
    try:
        rows = {method: robot_run(name, method, swap) for method in ('bfs', 'dfs')}
    except NotImplementedError as exc:
        print(f'PENDING: finish {exc or "your pasted Part 1 code"}\n')
        return
    print(f"{'Robot':<7}{'Route (steps)':>15}{'Cells explored':>16}{'Walked (steps)':>16}")
    for method, row in rows.items():
        print(f"{method.upper():<7}{row['route']:>15,}{row['explored']:>16,}{row['walked']:>16,}")
    print()

### Scenario 1 — The Warehouse (Many Routes)

On Map A, the robot must fetch a box from G, only four steps below S.

**Task:** Run the next cell. It compares the two robots, then swaps S and G and compares them
again.

**Requirements:**

- Record the route length, cells explored and walking distance of each robot, in both runs.
- Explain why the BFS robot's route is the shortest possible, and why the DFS robot's is not.
- Explain why swapping S and G changes the DFS robot's result so much, but leaves the length
  of the BFS robot's route unchanged. Hint: in what order does `incident_edges` list the
  neighbours of a cell built by your `build_maze_graph`?

In [34]:
compare_robots('warehouse')
compare_robots('warehouse', swap=True)

Map A: warehouse: from (1, 1) to (5, 1)
Robot    Route (steps)  Cells explored  Walked (steps)
BFS                  4              11              40
DFS                 60              70              70

Map A: warehouse: from (5, 1) to (1, 1)  (S and G swapped)
Robot    Route (steps)  Cells explored  Walked (steps)
BFS                  4              13              44
DFS                  4               5               4



**Your answer**

- **Metrics (both runs):**
  - **S (1, 1) to G (5, 1):** BFS — route 4 steps, 11 cells explored, 40 walked; DFS — route 60 steps, 70 cells explored, 70 walked.
  - **Swapped, G (5, 1) to S (1, 1):** BFS — route 4 steps, 13 cells explored, 44 walked; DFS — route 4 steps, 5 cells explored, 4 walked.
- **Why BFS is shortest and DFS is not:** BFS expands layer by layer in order of edge distance from the start, guaranteeing the shortest path (4 steps) on first　reaching G. DFS dives deep along one branch before backtracking, returning the first path it happens to find (60 steps) rather than the shortest.
- **Why swapping S and G changes DFS, not BFS length:** DFS depends on the neighbour order returned by incident_edges (set when build_maze_graph inserts edges): starting at (1, 1) sends DFS wandering through the warehouse (60 steps), whereas starting at　(5, 1) leads directly to (1, 1) (4 steps). BFS always finds a shortest path, and because the graph is undirected, the shortest distance is symmetric (4 steps both ways).

### Scenario 2 — The Maze (One Route)

On Map B, G is in the far corner. A connected graph with n vertices needs at least n − 1
edges, and every extra edge closes a loop, so m − n + 1 counts the loops. Zero loops means
exactly one route between any two cells.

**Task:** Run the next cell. It prints the profile of both maps, then compares the two robots
on Map B.

**Requirements:**

- Record m − n + 1 for both maps, and the three numbers of each robot on Map B.
- Explain why both robots found routes of the same length.
- Explain why the BFS robot walks several times as far as the DFS robot, although it explores
  fewer cells.

In [35]:
print_profiles()
compare_robots('maze')

Map                    n     m  m - n + 1  dead ends  junctions
Map A: warehouse     263   340         78          0        114
Map B: maze          209   208          0         12         10

Map B: maze: from (1, 1) to (13, 29)
Robot    Route (steps)  Cells explored  Walked (steps)
BFS                100             139           1,066
DFS                100             177             252



**Your answer**

- **Metrics:**
  - **Cyclomatic number ($m - n + 1$):** Map A (warehouse) = 78; Map B (maze) = 0.
  - **Map B ((1, 1) to (13, 29)):** BFS — route 100 steps, 139 cells explored, 1,066 steps walked; DFS — route 100 steps, 177 cells explored, 252 steps walked.
  
- **Why both robots found routes of the same length:** Map B has $m - n + 1 = 0$ loops, meaning it is a tree (acyclic connected graph) with a unique simple path between any two cells. Since there is only one possible route from S to G (100 steps), both BFS and DFS must return that exact same route length.
- **Why BFS walks several times as far as DFS despite exploring fewer cells:** Because a physical robot cannot teleport, it must walk along graph edges between consecutive visited cells. BFS expands level by level across separate branches, forcing the robot to repeatedly backtrack to earlier junctions and walk across the maze to reach the next frontier cell (1,066 steps), even though it stops as soon as it reaches G (139 cells). DFS follows one connected branch continuously into dead ends and only backtracks short distances (252 steps), even though it happens to explore more total cells (`177`) before finding G.

### Scenario 3 — Which Robot Would You Deploy?

The warehouse team will buy one search for each map. Use your numbers from Scenarios 1 and 2;
you do not need to run anything.

**Requirements:**

- For each map, choose BFS or DFS when the robot **already has the map** and only has to
  compute a route before it drives.
- For each map, choose BFS or DFS when the robot **has no map** and must explore by driving
  from cell to cell.
- Justify each choice with a number, and describe one situation in which one of your choices
  would be wrong.

**Your answer**

- Map A, a robot with the map and a robot without it, each with a number: with the map, BFS, because its route is 4 steps against DFS 60 steps. Without the map, BFS, because it walks 40 steps against DFS 70 steps and explores only 11 cells against 70.
- Map B, a robot with the map and a robot without it, each with a number: with the map, BFS, because both routes are 100 steps but BFS explores fewer cells, 139 against 177. Without the map, DFS, because it walks 252 steps against BFS 1,066 steps.
- One situation in which one of your choices would be wrong: on Map A without the map, swapping S and G makes DFS walk only 4 steps while BFS walks 44 steps, so choosing BFS would be wrong in that direction; the DFS result depends on which neighbour incident_edges lists first. More generally, a choice based on these two maps would be wrong on a maze with long straight corridors leading to dead ends (where DFS can waste steps exploring deep dead ends) or on a map where the goal is far from the start in a tree-like layout where BFS must repeatedly backtrack across wide branches.

## Check your implementation

Run the next cell after editing any function. It tests your pasted adjacency map and
traversals on small directed and undirected graphs, then your three Part 2 functions on small
grids and on both maps: one vertex per open cell, one edge per pair of neighbours, the order
of the neighbours, unchanged inputs, stopping at the goal, an unreachable goal and a bad
`method`.

Incorrect functions say FAILED; unfinished TODOs say PENDING. Neither is a pass.
**ALL CHECKS PASSED** appears only when every group passes.

In [36]:
FORBIDDEN = {'deque', 'heapq', 'Queue', 'LifoQueue', '_outgoing', '_incoming'}
CHECKED_FUNCTIONS = ('build_maze_graph', 'explore_until_goal', 'walking_distance')


class replace_global:
    """Temporarily replace a notebook-level function; your code finds it by name."""

    def __init__(self, name, replacement):
        self.name, self.replacement = name, replacement

    def __enter__(self):
        self.saved = globals()[self.name]
        globals()[self.name] = self.replacement

    def __exit__(self, *exc_info):
        globals()[self.name] = self.saved


def _names_used(code):
    """Every global or attribute name a function uses, including inner functions."""
    names = set(code.co_names)
    for constant in code.co_consts:
        if inspect.iscode(constant):
            names |= _names_used(constant)
    return names


def _raises(error, action, message):
    try:
        action()
    except error:
        return
    raise AssertionError(message)


def _sample_graph(directed=False):
    """A - B - C, A - D, B - E, D - E, E - F: the graph of tests/test_graph_traversals.py."""
    graph = AdjacencyMapGraph(directed=directed)
    vertex = {name: graph.insert_vertex(name) for name in 'ABCDEF'}
    for a, b in ('AB', 'BC', 'AD', 'BE', 'DE', 'EF'):
        graph.insert_edge(vertex[a], vertex[b], a + b)
    return graph, vertex


def _check_adjacency_map():
    graph, v = _sample_graph()
    assert not graph.is_directed() and graph.vertex_count() == 6 and graph.edge_count() == 6
    ab = graph.get_edge(v['A'], v['B'])
    assert ab is not None and graph.get_edge(v['B'], v['A']) is ab, 'an undirected edge is found from both ends'
    assert graph.get_edge(v['A'], v['C']) is None
    assert ab.element() == 'AB' and set(ab.endpoints()) == {v['A'], v['B']} and ab.opposite(v['A']) is v['B']
    neighbours = [edge.opposite(v['B']).element() for edge in graph.incident_edges(v['B'])]
    assert neighbours == ['A', 'C', 'E'], \
        f'incident_edges(B) gave {neighbours}: it must list edges in the order they were inserted'
    assert graph.degree(v['B']) == 3 and len(set(graph.edges())) == 6
    _raises(ValueError, lambda: graph.insert_edge(v['B'], v['A']), 'joining the same two vertices twice must raise ValueError')
    assert graph.remove_edge(graph.get_edge(v['E'], v['F'])) == 'EF' and graph.edge_count() == 5
    assert graph.remove_vertex(v['B']) == 'B' and graph.vertex_count() == 5 and graph.edge_count() == 2, \
        'remove_vertex must also remove every edge of that vertex'
    assert graph.degree(v['A']) == 1 and graph.degree(v['F']) == 0
    directed = AdjacencyMapGraph(directed=True)
    a, b = directed.insert_vertex('a'), directed.insert_vertex('b')
    edge = directed.insert_edge(a, b, 'ab')
    assert directed.is_directed() and directed.get_edge(a, b) is edge and directed.get_edge(b, a) is None
    assert directed.degree(a) == 1 and directed.degree(a, outgoing=False) == 0
    assert list(directed.incident_edges(b, outgoing=False)) == [edge]


def _check_traversals():
    graph, v = _sample_graph()
    assert ''.join(cell.element() for cell in dfs(graph, v['A'])) == 'ABCEDF', 'dfs order on the test graph'
    assert ''.join(cell.element() for cell in bfs(graph, v['A'])) == 'ABDCEF', 'bfs order on the test graph'
    for find in (dfs_paths, bfs_paths):
        discovered = find(graph, v['A'])
        assert set(discovered) == {v[name] for name in 'BCDEF'}, f'{find.__name__} must map every vertex but start'
    path = construct_path(v['A'], v['F'], bfs_paths(graph, v['A']))
    assert [cell.element() for cell in path] == ['A', 'B', 'E', 'F'], 'construct_path with bfs_paths'
    assert construct_path(v['A'], v['A'], {}) == [v['A']]
    lonely = AdjacencyMapGraph()
    p, q = lonely.insert_vertex('p'), lonely.insert_vertex('q')
    assert list(bfs(lonely, p)) == [p] and construct_path(p, q, bfs_paths(lonely, p)) == []
    directed = AdjacencyMapGraph(directed=True)
    a, b, c = (directed.insert_vertex(name) for name in 'abc')
    directed.insert_edge(a, b)
    directed.insert_edge(c, a)
    assert list(dfs(directed, a)) == [a, b] and list(bfs(directed, a)) == [a, b], \
        'a directed search follows outgoing edges only'


def _check_build():
    graph, cells = build_maze_graph(['S.', '#G'])
    assert (graph.vertex_count(), graph.edge_count()) == (3, 2), \
        'one vertex per open cell, and one edge per pair of open cells that share a side'
    assert set(cells) == {(0, 0), (0, 1), (1, 1)}, "cells must hold every open cell, and no '#' cell"
    assert all(cells[cell].element() == cell for cell in cells), 'each vertex element must be its (row, col)'
    assert not graph.is_directed(), 'the robot can walk both ways: build an undirected graph'
    assert graph.get_edge(cells[(0, 1)], cells[(1, 1)]) is not None
    assert graph.get_edge(cells[(0, 0)], cells[(1, 1)]) is None, 'diagonal cells are not neighbours'
    floor = ['...', '...', '...']
    before = list(floor)
    graph, cells = build_maze_graph(floor)
    assert floor == before, 'build_maze_graph must not change grid'
    assert graph.edge_count() == 12, f'a 3 x 3 floor has 12 edges, not {graph.edge_count()}'
    centre = cells[(1, 1)]
    order = [edge.opposite(centre).element() for edge in graph.incident_edges(centre)]
    assert order == [(0, 1), (1, 0), (1, 2), (2, 1)], \
        f'the centre cell lists its neighbours as {order}: join each cell to the cell above, then to the cell on its left'
    assert build_maze_graph(['###'])[0].vertex_count() == 0
    for name, size in {'warehouse': (263, 340), 'maze': (209, 208)}.items():
        graph, _ = build_maze_graph(MAPS[name])
        assert (graph.vertex_count(), graph.edge_count()) == size, \
            f'{MAP_TITLES[name]} has {size[0]} open cells and {size[1]} edges'


def _counting(search, taken):
    """Wrap your bfs or dfs so that the check can count the cells it hands out."""
    def counted(graph, start):
        for cell in search(graph, start):
            taken.append(cell)
            yield cell
    return counted


def _check_explore():
    graph, cells = build_maze_graph(['S.#.', '..#G'])
    start, near, far = cells[(0, 0)], cells[(1, 0)], cells[(1, 3)]
    for method, search in (('bfs', bfs), ('dfs', dfs)):
        full = list(search(graph, start))
        order = explore_until_goal(graph, start, near, method)
        assert order == full[:full.index(near) + 1], \
            f"explore_until_goal(..., '{method}') must return your {method} order, up to and including goal"
        assert explore_until_goal(graph, start, start, method) == [start], 'if start is goal, return [start]'
        assert explore_until_goal(graph, start, far, method) == full, \
            'if goal cannot be reached, return every cell the search visits'
    _raises(ValueError, lambda: explore_until_goal(graph, start, near, 'astar'),
            "explore_until_goal must raise ValueError for a method other than 'bfs' or 'dfs'")
    grid = MAPS['warehouse']
    graph, cells = build_maze_graph(grid)
    start, goal = cells[find_cell(grid, 'S')], cells[find_cell(grid, 'G')]
    for method, search in (('bfs', bfs), ('dfs', dfs)):
        taken = []
        with replace_global(method, _counting(search, taken)):
            order = explore_until_goal(graph, start, goal, method)
        assert taken, f"explore_until_goal(..., '{method}') must call your {method}"
        assert len(taken) == len(order), \
            f'your {method} went on for {len(taken) - len(order)} cells after goal: stop the loop with break'
    assert (graph.vertex_count(), graph.edge_count()) == (263, 340), 'explore_until_goal must not change graph'


def _check_walking():
    graph, cells = build_maze_graph(['S.', '#G'])
    s, middle, g = cells[(0, 0)], cells[(0, 1)], cells[(1, 1)]
    assert walking_distance(graph, [s, middle, g]) == 2
    assert walking_distance(graph, [s, g, middle]) == 3, \
        'from S to G the robot must pass (0, 1), so that walk is 2 steps, not 1'
    assert walking_distance(graph, []) == 0 and walking_distance(graph, [s]) == 0, \
        'an order with fewer than two cells needs 0 steps'
    order = [s, g]
    assert walking_distance(graph, tuple(order)) == 2 and order == [s, g]
    assert {'bfs_paths', 'construct_path'} <= _names_used(walking_distance.__code__), \
        'walking_distance must use your bfs_paths and construct_path'


def _check_tools():
    for name in CHECKED_FUNCTIONS:
        names = _names_used(globals()[name].__code__)
        if 'NotImplementedError' in names:
            raise NotImplementedError(name)   # an empty TODO proves nothing
        used = names & FORBIDDEN
        assert not used, f'{name} uses {sorted(used)}: reuse your Part 1 code and the public graph methods'
    assert {'bfs', 'dfs'} <= _names_used(explore_until_goal.__code__), \
        'explore_until_goal must call your bfs and dfs'


def _grid_distance(grid, a, b):
    """Steps from cell a to cell b by a plain BFS on the grid itself, to check your robots."""
    from collections import deque
    distance, queue = {a: 0}, deque([a])
    while queue:
        r, c = queue.popleft()
        for cell in ((r - 1, c), (r, c - 1), (r, c + 1), (r + 1, c)):
            if cell not in distance and grid[cell[0]][cell[1]] != '#':
                distance[cell] = distance[(r, c)] + 1
                queue.append(cell)
    return distance[b]


def _check_both_maps():
    for name, grid in MAPS.items():
        status, traces = record_robots(name)
        assert status == 'ok', f'{MAP_TITLES[name]}: {status}'
        shortest = _grid_distance(grid, find_cell(grid, 'S'), find_cell(grid, 'G'))
        assert len(traces['bfs']['route']) - 1 == shortest, f'{MAP_TITLES[name]}: the BFS route must be shortest'
        assert len(traces['dfs']['route']) - 1 >= shortest


CHECK_GROUPS = [('adjacency_map.py', _check_adjacency_map), ('traversals.py', _check_traversals),
                ('build_maze_graph', _check_build), ('explore_until_goal', _check_explore),
                ('walking_distance', _check_walking), ('no shortcuts', _check_tools)]


def run_lab_checks():
    """Reusable for grading: PENDING and FAILED are never counted as passed."""
    results = {}
    for name, check in CHECK_GROUPS:
        try:
            check()
        except NotImplementedError as exc:
            results[name] = 'PENDING'
            print(f'PENDING | {name}: finish {exc or name}')
        except Exception as exc:
            results[name] = 'FAILED'
            print(f'FAILED  | {name}: {type(exc).__name__}: {exc}')
        else:
            results[name] = 'PASSED'
            print(f'PASSED  | {name}')
    # The full run on both maps is meaningful only after every group above passes.
    final = 'both maps, both robots'
    if all(value == 'PASSED' for value in results.values()):
        start = time.perf_counter()
        try:
            _check_both_maps()
        except Exception as exc:
            results[final] = 'FAILED'
            print(f'FAILED  | {final}: {type(exc).__name__}: {exc}')
        else:
            results[final] = 'PASSED'
            print(f'PASSED  | {final} ({time.perf_counter() - start:.1f} s)')
        if results[final] == 'PASSED':
            print('ALL CHECKS PASSED')
            return results
    else:
        results[final] = 'PENDING'
        print(f'PENDING | {final}: runs after every group above passes')
    print('Work remains: finish every TODO, then rerun its cell and this cell.')
    return results


CHECK_RESULTS = run_lab_checks()

PASSED  | adjacency_map.py
PASSED  | traversals.py
PASSED  | build_maze_graph
PASSED  | explore_until_goal
PASSED  | walking_distance
PASSED  | no shortcuts
PASSED  | both maps, both robots (0.6 s)
ALL CHECKS PASSED


## Deliverables

Submit one notebook, with the executed **ALL CHECKS PASSED** output and your answers. Each file
of a larger project is one section here:

| File | Description | In this notebook |
|---|---|---|
| `adjacency_map.py`, `traversals.py` | Your graph code, unchanged | Part 1 |
| `maze_graph.py` | `build_maze_graph` | Part 2, Task 2.1 |
| `robot_search.py` | `explore_until_goal` and `walking_distance` | Part 2, Tasks 2.2 and 2.3 |
| `results.md` | Your comparison of the two robots | The three **Your answer** cells |